# Exploratory Data Analysis: Temporal, Spatial, and Topic Modeling

This notebook performs comprehensive exploratory analysis on Waymo crash data:

**Section 1: Temporal & Spatial Analysis**
- Time of day patterns (rush hour vs off-peak)
- Day of week and seasonal trends
- Geographic clustering and hotspot identification
- Street-level analysis
- Correlation between time/location and crash severity

**Section 2: Topic Modeling**
- Latent Dirichlet Allocation (LDA) for crash narrative themes
- BERTopic for contextualized topic discovery
- Comparison of crash types across topics
- Temporal evolution of topics

## Setup and Dependencies

In [5]:
!pip install pandas numpy matplotlib seaborn plotly folium geopandas
!pip install scikit-learn nltk wordcloud
!pip install bertopic umap-learn hdbscan
!pip install pyLDAvis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, time
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully.")

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 31.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 10.1 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 24.4 MB/s  0:00:00 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [wordcloud]/9 [nltk]t-learn]
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.

## Load Data

In [7]:
# Load crash data
#df = pd.read_csv('../cleaned-data/waymo_crashes_cleaned.csv')  

df_loc = pd.read_csv('/Users/kateli/Desktop/Classes/COMM277T/bna-fuhgeddaboudit/cleaned-data/waymo_crashes_location.csv')

print(f"Total crashes: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")

# Display sample
df.head()

Total crashes: 1350

Columns: ['Report ID', 'Report Version', 'Reporting Entity', 'Crash With', 'SV Pre-Crash Movement', 'CP Pre-Crash Movement', 'SV Precrash Speed (MPH)', 'Narrative', 'Highest Injury Severity Alleged', 'Any Air Bags Deployed?', 'Was Any Vehicle Towed?', 'Were All Passengers Belted?', 'City', 'State', 'Incident Date', 'Incident Time (24:00)', 'Roadway Type', 'Weather - Clear', 'Weather - Cloudy', 'Weather - Rain', 'Weather - Snow', 'Weather - Severe Wind', 'Weather - Fog/Smoke/Haze', 'Data Period']


KeyError: 'Date'

---
# Section 1: Temporal & Spatial Analysis
---

## 1.1 Temporal Data Preparation

In [ ]:
# Parse datetime columns
# Adjust column names based on your data
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], errors='coerce')

# Extract temporal features
df['date'] = df['datetime'].dt.date
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['month_name'] = df['datetime'].dt.month_name()
df['day_of_week'] = df['datetime'].dt.day_name()
df['day_of_week_num'] = df['datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['hour'] = df['datetime'].dt.hour
df['is_weekend'] = df['day_of_week_num'].isin([5, 6])

# Create time period categories
def categorize_time_of_day(hour):
    if pd.isna(hour):
        return 'Unknown'
    elif 6 <= hour < 9:
        return 'Morning Rush (6-9 AM)'
    elif 9 <= hour < 12:
        return 'Late Morning (9 AM-12 PM)'
    elif 12 <= hour < 15:
        return 'Midday (12-3 PM)'
    elif 15 <= hour < 18:
        return 'Evening Rush (3-6 PM)'
    elif 18 <= hour < 22:
        return 'Evening (6-10 PM)'
    else:
        return 'Night (10 PM-6 AM)'

df['time_period'] = df['hour'].apply(categorize_time_of_day)

# Season categorization
def get_season(month):
    if pd.isna(month):
        return 'Unknown'
    elif month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

print("Temporal features created.")
print(f"\nDate range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Time span: {(df['datetime'].max() - df['datetime'].min()).days} days")

## 1.2 Time of Day Analysis

In [ ]:
# Crashes by hour of day
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Hourly distribution
hourly_counts = df['hour'].value_counts().sort_index()
axes[0, 0].bar(hourly_counts.index, hourly_counts.values, color='steelblue', alpha=0.8)
axes[0, 0].axvspan(6, 9, alpha=0.2, color='orange', label='Morning Rush')
axes[0, 0].axvspan(15, 18, alpha=0.2, color='red', label='Evening Rush')
axes[0, 0].set_xlabel('Hour of Day', fontsize=12)
axes[0, 0].set_ylabel('Number of Crashes', fontsize=12)
axes[0, 0].set_title('Crashes by Hour of Day', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)
axes[0, 0].set_xticks(range(0, 24))

# 2. Time period distribution
time_period_order = [
    'Morning Rush (6-9 AM)', 
    'Late Morning (9 AM-12 PM)', 
    'Midday (12-3 PM)',
    'Evening Rush (3-6 PM)', 
    'Evening (6-10 PM)', 
    'Night (10 PM-6 AM)'
]
time_counts = df['time_period'].value_counts().reindex(time_period_order)
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#ff99cc', '#c2c2f0']
axes[0, 1].barh(range(len(time_counts)), time_counts.values, color=colors, alpha=0.8)
axes[0, 1].set_yticks(range(len(time_counts)))
axes[0, 1].set_yticklabels(time_counts.index)
axes[0, 1].set_xlabel('Number of Crashes', fontsize=12)
axes[0, 1].set_title('Crashes by Time Period', fontsize=14, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# Add percentages
for i, v in enumerate(time_counts.values):
    axes[0, 1].text(v + 2, i, f'{v} ({v/len(df)*100:.1f}%)', va='center')

# 3. Weekday vs Weekend by hour
weekday_hourly = df[~df['is_weekend']].groupby('hour').size()
weekend_hourly = df[df['is_weekend']].groupby('hour').size()

x = range(24)
width = 0.35
axes[1, 0].bar([i - width/2 for i in x], weekday_hourly.reindex(x, fill_value=0), 
               width, label='Weekday', alpha=0.8, color='steelblue')
axes[1, 0].bar([i + width/2 for i in x], weekend_hourly.reindex(x, fill_value=0), 
               width, label='Weekend', alpha=0.8, color='coral')
axes[1, 0].set_xlabel('Hour of Day', fontsize=12)
axes[1, 0].set_ylabel('Number of Crashes', fontsize=12)
axes[1, 0].set_title('Weekday vs Weekend Crashes by Hour', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].set_xticks(range(0, 24))

# 4. Rush hour analysis
rush_hour_data = [
    ('Morning Rush\n(6-9 AM)', len(df[(df['hour'] >= 6) & (df['hour'] < 9)])),
    ('Evening Rush\n(3-6 PM)', len(df[(df['hour'] >= 15) & (df['hour'] < 18)])),
    ('Other Times', len(df[~((df['hour'] >= 6) & (df['hour'] < 9)) & 
                             ~((df['hour'] >= 15) & (df['hour'] < 18))]))
]
labels, values = zip(*rush_hour_data)
colors_pie = ['#ff9999', '#ffcc99', '#99ccff']
explode = (0.05, 0.05, 0)
axes[1, 1].pie(values, labels=labels, autopct='%1.1f%%', startangle=90,
               colors=colors_pie, explode=explode, textprops={'fontsize': 11})
axes[1, 1].set_title('Rush Hour vs Other Times', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/temporal_analysis_hourly.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("\n" + "="*60)
print("TIME OF DAY ANALYSIS SUMMARY")
print("="*60)
print(f"\nPeak crash hour: {hourly_counts.idxmax()}:00 ({hourly_counts.max()} crashes)")
print(f"Lowest crash hour: {hourly_counts.idxmin()}:00 ({hourly_counts.min()} crashes)")
print(f"\nMorning rush (6-9 AM): {rush_hour_data[0][1]} crashes ({rush_hour_data[0][1]/len(df)*100:.1f}%)")
print(f"Evening rush (3-6 PM): {rush_hour_data[1][1]} crashes ({rush_hour_data[1][1]/len(df)*100:.1f}%)")
print(f"Total rush hour crashes: {rush_hour_data[0][1] + rush_hour_data[1][1]} ({(rush_hour_data[0][1] + rush_hour_data[1][1])/len(df)*100:.1f}%)")

## 1.3 Day of Week and Seasonal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
colors_dow = ['steelblue']*5 + ['coral']*2  # Weekdays blue, weekends coral
axes[0, 0].bar(range(len(day_counts)), day_counts.values, color=colors_dow, alpha=0.8)
axes[0, 0].set_xticks(range(len(day_counts)))
axes[0, 0].set_xticklabels(day_counts.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Number of Crashes', fontsize=12)
axes[0, 0].set_title('Crashes by Day of Week', fontsize=14, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(day_counts.values):
    axes[0, 0].text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

# 2. Weekday vs Weekend
weekend_data = df['is_weekend'].value_counts()
labels = ['Weekday', 'Weekend']
values = [weekend_data[False], weekend_data[True]]
colors_we = ['steelblue', 'coral']
axes[0, 1].pie(values, labels=labels, autopct='%1.1f%%', startangle=90,
               colors=colors_we, explode=(0.05, 0), textprops={'fontsize': 12})
axes[0, 1].set_title('Weekday vs Weekend Crashes', fontsize=14, fontweight='bold')

# 3. Monthly distribution
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
month_counts = df['month_name'].value_counts().reindex(month_order)
axes[1, 0].plot(range(len(month_counts)), month_counts.values, marker='o', 
                linewidth=2, markersize=8, color='steelblue')
axes[1, 0].fill_between(range(len(month_counts)), month_counts.values, alpha=0.3)
axes[1, 0].set_xticks(range(len(month_counts)))
axes[1, 0].set_xticklabels([m[:3] for m in month_counts.index], rotation=45, ha='right')
axes[1, 0].set_ylabel('Number of Crashes', fontsize=12)
axes[1, 0].set_title('Crashes by Month', fontsize=14, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Seasonal distribution
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_counts = df['season'].value_counts().reindex(season_order)
colors_season = ['#85C1E2', '#90EE90', '#FFD700', '#FF8C00']
axes[1, 1].bar(range(len(season_counts)), season_counts.values, 
               color=colors_season, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 1].set_xticks(range(len(season_counts)))
axes[1, 1].set_xticklabels(season_counts.index)
axes[1, 1].set_ylabel('Number of Crashes', fontsize=12)
axes[1, 1].set_title('Crashes by Season', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(season_counts.values):
    axes[1, 1].text(i, v + 1, f'{v}\n({v/len(df)*100:.1f}%)', 
                    ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/temporal_analysis_calendar.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary
print("\n" + "="*60)
print("CALENDAR PATTERN ANALYSIS")
print("="*60)
print(f"\nBusiest day: {day_counts.idxmax()} ({day_counts.max()} crashes)")
print(f"Quietest day: {day_counts.idxmin()} ({day_counts.min()} crashes)")
print(f"\nBusiest month: {month_counts.idxmax()} ({month_counts.max()} crashes)")
print(f"Quietest month: {month_counts.idxmin()} ({month_counts.min()} crashes)")
print(f"\nBusiest season: {season_counts.idxmax()} ({season_counts.max()} crashes)")

## 1.4 Temporal Heatmap

In [ ]:
# Create hour x day of week heatmap
heatmap_data = df.groupby(['day_of_week_num', 'hour']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='hour', columns='day_of_week_num', values='count').fillna(0)

# Reorder columns to match day order
heatmap_pivot.columns = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_pivot, cmap='YlOrRd', annot=True, fmt='.0f', 
            linewidths=0.5, cbar_kws={'label': 'Number of Crashes'})
plt.title('Crash Frequency Heatmap: Hour of Day × Day of Week', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Day of Week', fontsize=12, fontweight='bold')
plt.ylabel('Hour of Day', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/temporal_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Find peak time
peak_hour = heatmap_pivot.max(axis=1).idxmax()
peak_day = heatmap_pivot.max(axis=0).idxmax()
peak_value = heatmap_pivot.max().max()

print(f"\nPeak crash time: {peak_day} at {peak_hour}:00 ({int(peak_value)} crashes)")

## 1.5 Spatial Data Preparation

In [ ]:
# Extract location information from narratives or use location columns
# This depends on your data structure - adjust as needed

# Option 1: If you have explicit location columns
if 'Location' in df.columns:
    df['location_clean'] = df['Location'].str.strip()

# Option 2: Extract from narrative using regex
import re

def extract_street_names(narrative):
    """Extract street names from crash narratives."""
    if pd.isna(narrative):
        return []
    
    # Pattern for common street name formats
    # Note: This is simplified - adjust based on your data
    patterns = [
        r'on ([A-Z][a-z]+ (?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Way|Lane|Ln))',
        r'at ([A-Z][a-z]+ (?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Way|Lane|Ln))',
        r'intersection (?:of|at) ([A-Z][a-z]+ (?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd))',
    ]
    
    streets = []
    for pattern in patterns:
        matches = re.findall(pattern, narrative)
        streets.extend(matches)
    
    return streets

# Extract streets (if narratives are available)
if 'Narrative' in df.columns:
    df['streets_mentioned'] = df['Narrative'].apply(extract_street_names)
    df['num_streets'] = df['streets_mentioned'].apply(len)
    df['primary_street'] = df['streets_mentioned'].apply(lambda x: x[0] if x else None)

# Option 3: If you have coordinates
if 'Latitude' in df.columns and 'Longitude' in df.columns:
    df['has_coordinates'] = ~df['Latitude'].isna() & ~df['Longitude'].isna()
    print(f"Crashes with coordinates: {df['has_coordinates'].sum()} ({df['has_coordinates'].sum()/len(df)*100:.1f}%)")

# Extract city/region if available
if 'City' in df.columns:
    df['city_clean'] = df['City'].str.strip().str.title()
elif 'Narrative' in df.columns:
    # Try to extract city from narrative
    def extract_city(narrative):
        cities = ['San Francisco', 'Phoenix', 'Los Angeles', 'Austin']  # Add known cities
        if pd.isna(narrative):
            return None
        for city in cities:
            if city in narrative:
                return city
        return None
    
    df['city_clean'] = df['Narrative'].apply(extract_city)

print("\nSpatial data prepared.")
if 'city_clean' in df.columns:
    print(f"\nCrashes by city:")
    print(df['city_clean'].value_counts())

## 1.6 Geographic Distribution

In [ ]:
# City/Region distribution
if 'city_clean' in df.columns and df['city_clean'].notna().sum() > 0:
    city_counts = df['city_clean'].value_counts().head(10)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    axes[0].barh(range(len(city_counts)), city_counts.values, color='teal', alpha=0.8)
    axes[0].set_yticks(range(len(city_counts)))
    axes[0].set_yticklabels(city_counts.index)
    axes[0].set_xlabel('Number of Crashes', fontsize=12)
    axes[0].set_title('Top 10 Cities by Crash Count', fontsize=14, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Add percentages
    for i, v in enumerate(city_counts.values):
        axes[0].text(v + 1, i, f'{v} ({v/len(df)*100:.1f}%)', va='center')
    
    # Pie chart (top 5)
    top5 = city_counts.head(5)
    other_count = city_counts[5:].sum() if len(city_counts) > 5 else 0
    
    if other_count > 0:
        pie_labels = list(top5.index) + ['Other']
        pie_values = list(top5.values) + [other_count]
    else:
        pie_labels = list(top5.index)
        pie_values = list(top5.values)
    
    colors = plt.cm.Set3(range(len(pie_labels)))
    axes[1].pie(pie_values, labels=pie_labels, autopct='%1.1f%%', startangle=90,
                colors=colors, textprops={'fontsize': 10})
    axes[1].set_title('Geographic Distribution (Top 5 Cities)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../figures/spatial_city_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("City information not available in dataset.")

## 1.7 Street-Level Analysis

In [ ]:
# Analyze most common crash locations (streets/intersections)
if 'primary_street' in df.columns:
    street_counts = df['primary_street'].value_counts().head(20)
    
    plt.figure(figsize=(14, 8))
    plt.barh(range(len(street_counts)), street_counts.values, color='darkblue', alpha=0.7)
    plt.yticks(range(len(street_counts)), street_counts.index, fontsize=10)
    plt.xlabel('Number of Crashes', fontsize=12, fontweight='bold')
    plt.title('Top 20 Streets by Crash Frequency', fontsize=16, fontweight='bold', pad=20)
    plt.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(street_counts.values):
        plt.text(v + 0.2, i, str(v), va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../figures/spatial_street_hotspots.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("STREET-LEVEL HOTSPOT ANALYSIS")
    print("="*60)
    print(f"\nTop crash location: {street_counts.index[0]} ({street_counts.values[0]} crashes)")
    print(f"\nTop 5 streets account for {street_counts.head(5).sum()} crashes ({street_counts.head(5).sum()/len(df)*100:.1f}% of total)")
else:
    print("Street information could not be extracted from data.")

## 1.8 Interactive Map Visualization (if coordinates available)

In [ ]:
# Interactive map with Plotly (if coordinates available)
if 'Latitude' in df.columns and 'Longitude' in df.columns:
    df_map = df[df['has_coordinates']].copy()
    
    if len(df_map) > 0:
        # Create hover text
        df_map['hover_text'] = (
            'Date: ' + df_map['date'].astype(str) + '<br>' +
            'Time: ' + df_map['hour'].astype(str) + ':00<br>' +
            'Day: ' + df_map['day_of_week'] + '<br>'
        )
        
        # Create map
        fig = px.scatter_mapbox(
            df_map,
            lat='Latitude',
            lon='Longitude',
            color='time_period',
            hover_data=['hover_text'],
            zoom=10,
            height=600,
            title='Geographic Distribution of Crashes by Time Period'
        )
        
        fig.update_layout(
            mapbox_style='open-street-map',
            margin={"r":0,"t":50,"l":0,"b":0}
        )
        
        fig.show()
        fig.write_html('../figures/crash_map_interactive.html')
        print("Interactive map saved to: ../figures/crash_map_interactive.html")
    else:
        print("No valid coordinates found in dataset.")
else:
    print("Coordinate data not available for mapping.")

## 1.9 Temporal-Spatial Cross-Analysis

In [ ]:
# Analyze crash patterns by city and time period
if 'city_clean' in df.columns and df['city_clean'].notna().sum() > 0:
    top_cities = df['city_clean'].value_counts().head(3).index
    df_top_cities = df[df['city_clean'].isin(top_cities)]
    
    # Time of day patterns by city
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Hourly patterns by city
    for city in top_cities:
        city_data = df_top_cities[df_top_cities['city_clean'] == city]
        hourly = city_data['hour'].value_counts().sort_index()
        axes[0].plot(hourly.index, hourly.values, marker='o', label=city, linewidth=2)
    
    axes[0].set_xlabel('Hour of Day', fontsize=12)
    axes[0].set_ylabel('Number of Crashes', fontsize=12)
    axes[0].set_title('Hourly Crash Patterns by City', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(range(0, 24))
    
    # 2. Day of week patterns by city
    city_day_data = df_top_cities.groupby(['city_clean', 'day_of_week']).size().reset_index(name='count')
    city_day_pivot = city_day_data.pivot(index='day_of_week', columns='city_clean', values='count')
    city_day_pivot = city_day_pivot.reindex(day_order)
    
    city_day_pivot.plot(kind='bar', ax=axes[1], width=0.8)
    axes[1].set_xlabel('Day of Week', fontsize=12)
    axes[1].set_ylabel('Number of Crashes', fontsize=12)
    axes[1].set_title('Day of Week Patterns by City', fontsize=14, fontweight='bold')
    axes[1].set_xticklabels(city_day_pivot.index, rotation=45, ha='right')
    axes[1].legend(title='City')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../figures/temporal_spatial_cross_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 1.10 Correlation with Crash Severity

In [ ]:
# Analyze if time/location correlates with crash severity
# This requires a severity indicator - adjust based on your data

# Option 1: Use injury indicator if available
if 'Injury' in df.columns or 'Injury Reported' in df.columns:
    injury_col = 'Injury' if 'Injury' in df.columns else 'Injury Reported'
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Injury rate by time of day
    injury_by_hour = df.groupby('hour')[injury_col].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100 if len(x) > 0 else 0
    )
    
    axes[0].plot(injury_by_hour.index, injury_by_hour.values, 
                 marker='o', linewidth=2, color='red', markersize=8)
    axes[0].fill_between(injury_by_hour.index, injury_by_hour.values, alpha=0.3, color='red')
    axes[0].set_xlabel('Hour of Day', fontsize=12)
    axes[0].set_ylabel('Injury Rate (%)', fontsize=12)
    axes[0].set_title('Injury Rate by Hour of Day', fontsize=14, fontweight='bold')
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(range(0, 24))
    
    # Injury rate by day of week
    injury_by_day = df.groupby('day_of_week')[injury_col].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100 if len(x) > 0 else 0
    ).reindex(day_order)
    
    axes[1].bar(range(len(injury_by_day)), injury_by_day.values, 
                color='darkred', alpha=0.7)
    axes[1].set_xticks(range(len(injury_by_day)))
    axes[1].set_xticklabels(injury_by_day.index, rotation=45, ha='right')
    axes[1].set_ylabel('Injury Rate (%)', fontsize=12)
    axes[1].set_title('Injury Rate by Day of Week', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../figures/severity_temporal_correlation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("SEVERITY CORRELATION ANALYSIS")
    print("="*60)
    print(f"\nHighest injury rate hour: {injury_by_hour.idxmax()}:00 ({injury_by_hour.max():.1f}%)")
    print(f"Lowest injury rate hour: {injury_by_hour.idxmin()}:00 ({injury_by_hour.min():.1f}%)")
    print(f"\nHighest injury rate day: {injury_by_day.idxmax()} ({injury_by_day.max():.1f}%)")
    print(f"Lowest injury rate day: {injury_by_day.idxmin()} ({injury_by_day.min():.1f}%)")

---
# Section 2: Topic Modeling
---

## 2.1 Text Preprocessing

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

def preprocess_text(text):
    """Preprocess text for topic modeling."""
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and short words
    stop_words = set(stopwords.words('english'))
    # Add domain-specific stopwords
    custom_stopwords = {'waymo', 'av', 'vehicle', 'xxx', 'time', 'impact', 'level', 'ads', 'engaged'}
    stop_words.update(custom_stopwords)
    
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return ' '.join(tokens)

# Preprocess narratives
if 'Narrative' in df.columns:
    print("Preprocessing narratives for topic modeling...")
    df['narrative_processed'] = df['Narrative'].apply(preprocess_text)
    
    # Remove empty narratives
    df_topics = df[df['narrative_processed'].str.len() > 10].copy()
    
    print(f"\nDocuments for topic modeling: {len(df_topics)}")
    print(f"\nExample processed text:")
    print(df_topics['narrative_processed'].iloc[0][:200])
else:
    print("No narrative column found. Topic modeling requires text data.")

## 2.2 Latent Dirichlet Allocation (LDA)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud

if 'narrative_processed' in df_topics.columns:
    # Create document-term matrix
    print("Creating document-term matrix...")
    vectorizer = CountVectorizer(
        max_features=1000,  # Top 1000 words
        min_df=5,          # Word must appear in at least 5 documents
        max_df=0.7,        # Word must appear in less than 70% of documents
        ngram_range=(1, 2) # Include bigrams
    )
    
    doc_term_matrix = vectorizer.fit_transform(df_topics['narrative_processed'])
    feature_names = vectorizer.get_feature_names_out()
    
    print(f"Document-term matrix shape: {doc_term_matrix.shape}")
    print(f"Vocabulary size: {len(feature_names)}")
    
    # Train LDA model
    print("\nTraining LDA model...")
    n_topics = 8  # Adjust based on your data
    
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        max_iter=50,
        learning_method='batch',
        random_state=42,
        n_jobs=-1
    )
    
    lda_topics = lda_model.fit_transform(doc_term_matrix)
    
    print("LDA training complete.")
    
    # Display topics
    def display_topics(model, feature_names, n_top_words=10):
        """Display top words for each topic."""
        topics = {}
        for topic_idx, topic in enumerate(model.components_):
            top_indices = topic.argsort()[-n_top_words:][::-1]
            top_words = [feature_names[i] for i in top_indices]
            topics[f'Topic {topic_idx + 1}'] = top_words
        return topics
    
    topics_dict = display_topics(lda_model, feature_names, n_top_words=12)
    
    print("\n" + "="*80)
    print("LDA TOPIC MODELING RESULTS")
    print("="*80)
    for topic_name, words in topics_dict.items():
        print(f"\n{topic_name}: {', '.join(words)}")

## 2.3 Topic Interpretation and Labeling

In [ ]:
# Assign interpretive labels to topics based on top words
# ADJUST THESE LABELS based on the actual topics discovered
topic_labels = {
    0: 'Rear-end Collisions',
    1: 'Intersection Incidents',
    2: 'Lane Change/Sideswipe',
    3: 'Pedestrian/Cyclist Interactions',
    4: 'Parking/Stationary Incidents',
    5: 'Door Opening Collisions',
    6: 'Traffic Signal/Stop Sign',
    7: 'Object/Animal Collisions'
}

# Assign dominant topic to each document
df_topics['dominant_topic'] = lda_topics.argmax(axis=1)
df_topics['dominant_topic_prob'] = lda_topics.max(axis=1)
df_topics['topic_label'] = df_topics['dominant_topic'].map(topic_labels)

# Topic distribution
topic_counts = df_topics['topic_label'].value_counts()

print("\nTopic Distribution:")
for topic, count in topic_counts.items():
    print(f"{topic}: {count} ({count/len(df_topics)*100:.1f}%)")

## 2.4 Topic Visualization

In [ ]:
# Visualize topic distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Topic frequency bar chart
topic_counts_sorted = topic_counts.sort_values(ascending=True)
colors = plt.cm.Set3(range(len(topic_counts_sorted)))
axes[0, 0].barh(range(len(topic_counts_sorted)), topic_counts_sorted.values, 
                color=colors, alpha=0.8)
axes[0, 0].set_yticks(range(len(topic_counts_sorted)))
axes[0, 0].set_yticklabels(topic_counts_sorted.index, fontsize=10)
axes[0, 0].set_xlabel('Number of Crashes', fontsize=12)
axes[0, 0].set_title('Topic Distribution', fontsize=14, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Add percentages
for i, v in enumerate(topic_counts_sorted.values):
    axes[0, 0].text(v + 5, i, f'{v} ({v/len(df_topics)*100:.1f}%)', va='center')

# 2. Topic confidence distribution
axes[0, 1].hist(df_topics['dominant_topic_prob'], bins=30, color='steelblue', 
                alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Topic Assignment Probability', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Topic Assignment Confidence', fontsize=14, fontweight='bold')
axes[0, 1].axvline(df_topics['dominant_topic_prob'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean: {df_topics["dominant_topic_prob"].mean():.2f}')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Topics over time (if temporal data available)
if 'datetime' in df_topics.columns:
    # Aggregate by month
    df_topics['year_month'] = df_topics['datetime'].dt.to_period('M')
    topic_time = df_topics.groupby(['year_month', 'topic_label']).size().reset_index(name='count')
    topic_time_pivot = topic_time.pivot(index='year_month', columns='topic_label', values='count').fillna(0)
    
    # Plot top 5 topics
    top_5_topics = topic_counts.head(5).index
    for topic in top_5_topics:
        if topic in topic_time_pivot.columns:
            axes[1, 0].plot(range(len(topic_time_pivot)), topic_time_pivot[topic].values, 
                           marker='o', label=topic, linewidth=2)
    
    axes[1, 0].set_xlabel('Time Period', fontsize=12)
    axes[1, 0].set_ylabel('Number of Crashes', fontsize=12)
    axes[1, 0].set_title('Topic Trends Over Time (Top 5)', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=8, loc='best')
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].set_xticks([])

# 4. Topic proportions pie chart (top 6)
top_6_topics = topic_counts.head(6)
other_count = topic_counts[6:].sum() if len(topic_counts) > 6 else 0

if other_count > 0:
    pie_labels = list(top_6_topics.index) + ['Other']
    pie_values = list(top_6_topics.values) + [other_count]
else:
    pie_labels = list(top_6_topics.index)
    pie_values = list(top_6_topics.values)

colors_pie = plt.cm.Set3(range(len(pie_labels)))
axes[1, 1].pie(pie_values, labels=pie_labels, autopct='%1.1f%%', startangle=90,
               colors=colors_pie, textprops={'fontsize': 9})
axes[1, 1].set_title('Topic Proportions (Top 6)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/topic_modeling_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.5 Word Clouds for Each Topic

In [ ]:
# Generate word clouds for each topic
n_topics_display = min(8, n_topics)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for topic_idx in range(n_topics_display):
    # Get top words and their weights
    topic = lda_model.components_[topic_idx]
    top_indices = topic.argsort()[-30:][::-1]
    top_words = {feature_names[i]: topic[i] for i in top_indices}
    
    # Generate word cloud
    wordcloud = WordCloud(
        width=400, 
        height=300,
        background_color='white',
        colormap='viridis',
        relative_scaling=0.5
    ).generate_from_frequencies(top_words)
    
    # Plot
    axes[topic_idx].imshow(wordcloud, interpolation='bilinear')
    axes[topic_idx].axis('off')
    topic_name = topic_labels.get(topic_idx, f'Topic {topic_idx + 1}')
    count = len(df_topics[df_topics['dominant_topic'] == topic_idx])
    axes[topic_idx].set_title(f'{topic_name}\n({count} crashes)', 
                              fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/topic_wordclouds.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.6 BERTopic for Contextualized Topics

In [ ]:
# BERTopic for more sophisticated topic modeling
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    
    print("Training BERTopic model...")
    print("This may take several minutes...\n")
    
    # Initialize BERTopic
    topic_model = BERTopic(
        language='english',
        calculate_probabilities=True,
        verbose=True,
        min_topic_size=10  # Adjust based on your dataset size
    )
    
    # Fit model
    topics, probs = topic_model.fit_transform(df_topics['Narrative'].tolist())
    
    # Add to dataframe
    df_topics['bertopic'] = topics
    df_topics['bertopic_prob'] = probs.max(axis=1) if probs is not None else 0
    
    # Get topic info
    topic_info = topic_model.get_topic_info()
    
    print("\n" + "="*80)
    print("BERTOPIC RESULTS")
    print("="*80)
    print(f"\nNumber of topics discovered: {len(topic_info) - 1}")  # -1 for outlier topic
    print("\nTop 10 topics:")
    print(topic_info.head(10))
    
    # Visualize topics
    fig = topic_model.visualize_topics()
    fig.write_html('../figures/bertopic_visualization.html')
    print("\nBERTopic visualization saved to: ../figures/bertopic_visualization.html")
    
    # Visualize topic hierarchy
    fig_hierarchy = topic_model.visualize_hierarchy()
    fig_hierarchy.write_html('../figures/bertopic_hierarchy.html')
    print("Topic hierarchy saved to: ../figures/bertopic_hierarchy.html")
    
except ImportError:
    print("BERTopic not installed. Install with: pip install bertopic")
except Exception as e:
    print(f"Error in BERTopic analysis: {e}")

## 2.7 Topic-Temporal Cross-Analysis

In [ ]:
# Analyze topic distribution by time of day
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Topics by time period
topic_time_period = pd.crosstab(df_topics['time_period'], df_topics['topic_label'], 
                                 normalize='index') * 100
topic_time_period = topic_time_period.reindex(time_period_order)

top_4_topics = topic_counts.head(4).index
topic_time_period[top_4_topics].plot(kind='bar', ax=axes[0, 0], width=0.8)
axes[0, 0].set_xlabel('Time Period', fontsize=12)
axes[0, 0].set_ylabel('Percentage of Crashes (%)', fontsize=12)
axes[0, 0].set_title('Topic Distribution by Time of Day', fontsize=14, fontweight='bold')
axes[0, 0].legend(title='Topic', fontsize=8, loc='upper right')
axes[0, 0].set_xticklabels(topic_time_period.index, rotation=45, ha='right')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Topics by day of week
topic_dow = pd.crosstab(df_topics['day_of_week'], df_topics['topic_label'], 
                        normalize='index') * 100
topic_dow = topic_dow.reindex(day_order)

topic_dow[top_4_topics].plot(kind='bar', ax=axes[0, 1], width=0.8)
axes[0, 1].set_xlabel('Day of Week', fontsize=12)
axes[0, 1].set_ylabel('Percentage of Crashes (%)', fontsize=12)
axes[0, 1].set_title('Topic Distribution by Day of Week', fontsize=14, fontweight='bold')
axes[0, 1].legend(title='Topic', fontsize=8, loc='upper right')
axes[0, 1].set_xticklabels(topic_dow.index, rotation=45, ha='right')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Heatmap: Topic vs Hour
topic_hour_heatmap = pd.crosstab(df_topics['hour'], df_topics['topic_label'])
sns.heatmap(topic_hour_heatmap[top_4_topics], cmap='YlOrRd', annot=True, 
            fmt='d', ax=axes[1, 0], cbar_kws={'label': 'Count'})
axes[1, 0].set_xlabel('Topic', fontsize=12)
axes[1, 0].set_ylabel('Hour of Day', fontsize=12)
axes[1, 0].set_title('Topic × Hour Heatmap (Top 4 Topics)', fontsize=14, fontweight='bold')

# 4. Weekend vs Weekday topic differences
topic_weekend = pd.crosstab(df_topics['is_weekend'], df_topics['topic_label'], 
                            normalize='index') * 100
topic_weekend.index = ['Weekday', 'Weekend']

topic_weekend.T.plot(kind='barh', ax=axes[1, 1], width=0.7)
axes[1, 1].set_xlabel('Percentage (%)', fontsize=12)
axes[1, 1].set_ylabel('Topic', fontsize=12)
axes[1, 1].set_title('Weekend vs Weekday Topic Distribution', fontsize=14, fontweight='bold')
axes[1, 1].legend(title='Period')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/topic_temporal_cross_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.8 Representative Examples for Each Topic

In [ ]:
# Show representative examples for each topic
print("\n" + "="*80)
print("REPRESENTATIVE CRASH NARRATIVES BY TOPIC")
print("="*80)

for topic_idx in range(min(n_topics, 5)):  # Show first 5 topics
    topic_name = topic_labels.get(topic_idx, f'Topic {topic_idx + 1}')
    topic_docs = df_topics[df_topics['dominant_topic'] == topic_idx]
    
    # Get documents with highest topic probability
    top_docs = topic_docs.nlargest(2, 'dominant_topic_prob')
    
    print(f"\n{'-'*80}")
    print(f"{topic_name} ({len(topic_docs)} crashes)")
    print(f"{'-'*80}")
    
    for idx, row in top_docs.iterrows():
        print(f"\nExample {idx}:")
        print(f"Probability: {row['dominant_topic_prob']:.3f}")
        print(f"Date: {row['date'] if 'date' in row else 'N/A'}")
        print(f"Time: {row['time_period'] if 'time_period' in row else 'N/A'}")
        print(f"Narrative: {row['Narrative'][:300]}...")
        print()

## 2.9 Topic-Spatial Analysis

In [ ]:
# Analyze topic distribution by location (if available)
if 'city_clean' in df_topics.columns and df_topics['city_clean'].notna().sum() > 0:
    top_cities = df_topics['city_clean'].value_counts().head(3).index
    df_top_cities_topics = df_topics[df_topics['city_clean'].isin(top_cities)]
    
    # Topic distribution by city
    topic_city = pd.crosstab(df_top_cities_topics['city_clean'], 
                             df_top_cities_topics['topic_label'], 
                             normalize='index') * 100
    
    plt.figure(figsize=(14, 8))
    topic_city.plot(kind='bar', width=0.8, figsize=(14, 8))
    plt.xlabel('City', fontsize=12, fontweight='bold')
    plt.ylabel('Percentage of Crashes (%)', fontsize=12, fontweight='bold')
    plt.title('Topic Distribution by City', fontsize=16, fontweight='bold', pad=20)
    plt.legend(title='Topic', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../figures/topic_spatial_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

## 2.10 Summary Statistics and Insights

In [ ]:
# Comprehensive summary
print("\n" + "="*80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

print("\n[TEMPORAL PATTERNS]")
print(f"• Peak crash hour: {df['hour'].mode()[0]}:00")
print(f"• Rush hour crashes: {(df['hour'].between(6, 9) | df['hour'].between(15, 18)).sum()} ({(df['hour'].between(6, 9) | df['hour'].between(15, 18)).sum()/len(df)*100:.1f}%)")
print(f"• Busiest day: {df['day_of_week'].mode()[0]}")
print(f"• Weekend crashes: {df['is_weekend'].sum()} ({df['is_weekend'].sum()/len(df)*100:.1f}%)")

if 'city_clean' in df.columns:
    print("\n[SPATIAL PATTERNS]")
    top_city = df['city_clean'].mode()[0] if df['city_clean'].notna().sum() > 0 else 'N/A'
    print(f"• Primary location: {top_city}")
    if 'primary_street' in df.columns and df['primary_street'].notna().sum() > 0:
        print(f"• Most common crash location: {df['primary_street'].mode()[0]}")

print("\n[TOPIC MODELING]")
print(f"• Number of topics identified: {n_topics}")
print(f"• Most common crash type: {topic_counts.index[0]} ({topic_counts.values[0]} crashes)")
print(f"• Average topic assignment confidence: {df_topics['dominant_topic_prob'].mean():.2f}")

print("\n[KEY INSIGHTS]")
print("1. Temporal: Identify if crashes cluster during rush hours or specific days")
print("2. Spatial: Highlight geographic hotspots for targeted interventions")
print("3. Thematic: Common crash scenarios can inform safety improvements")
print("4. Cross-patterns: Certain crash types may be more prevalent at specific times/locations")

## 2.11 Export Results

In [ ]:
# Save augmented dataset with temporal, spatial, and topic features
df_topics.to_csv('../data/waymo_crashes_with_topics.csv', index=False)
print("Augmented dataset saved to: ../data/waymo_crashes_with_topics.csv")

# Save topic model
import pickle
with open('../models/lda_topic_model.pkl', 'wb') as f:
    pickle.dump(lda_model, f)
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Topic model saved to: ../models/lda_topic_model.pkl")
print("\nAnalysis complete!")